# Metrics Engine Demo

This notebook demonstrates the Week 2 portfolio risk engine on adjusted-close ETF data. Calculations are delegated to `mrrp.risk`; the notebook is responsible only for selecting a sample portfolio and presenting results.

In [ ]:
import sys
from pathlib import Path

import plotly.express as px

project_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(project_root / "src"))

In [ ]:
from mrrp.data.cache import load_parquet
from mrrp.risk.beta import rolling_beta
from mrrp.risk.drawdown import drawdown_series
from mrrp.risk.returns import cumulative_returns, portfolio_returns, simple_returns
from mrrp.risk.summary import portfolio_risk_summary
from mrrp.risk.volatility import rolling_volatility

## 1. Load Adjusted-Close Data

The notebook uses the parquet file produced by `make data`. If it is absent, run that command from the repository root before continuing.

In [ ]:
data_path = project_root / "data/processed/adjusted_close.parquet"
if not data_path.exists():
    raise FileNotFoundError(
        f"Adjusted-close data not found at {data_path}. Run `make data` first."
    )

prices = load_parquet(data_path)
prices.tail()

## 2. Define a Sample ETF Portfolio

The example combines U.S. growth, developed international, emerging-market, and Canadian equity ETFs. `SPY` is used only as the benchmark and is therefore excluded from portfolio weights.

In [ ]:
weights = {
    "QQQ": 0.35,
    "EFA": 0.25,
    "EEM": 0.15,
    "XIU.TO": 0.25,
}
benchmark = "SPY"
selected_tickers = [*weights, benchmark]

missing_tickers = [ticker for ticker in selected_tickers if ticker not in prices]
if missing_tickers:
    raise ValueError(f"Dataset is missing required tickers: {missing_tickers}")

analysis_prices = prices[selected_tickers].dropna()
analysis_prices.head()

## 3. Portfolio Risk Summary

The integrated summary returns numeric values without presentation formatting, making it suitable for downstream reporting or dashboards. VaR and CVaR remain signed returns rather than positive loss magnitudes.

In [ ]:
risk_summary = portfolio_risk_summary(
    prices=analysis_prices,
    weights=weights,
    benchmark=benchmark,
)
risk_summary

## 4. Build the Return Series for Charts

The same core return functions used by the summary provide a single portfolio return series and its aligned benchmark series for all charts below.

In [ ]:
asset_returns = simple_returns(analysis_prices)
portfolio_return_series = portfolio_returns(
    asset_returns[list(weights)],
    weights,
).dropna()
benchmark_return_series = asset_returns[benchmark].reindex(
    portfolio_return_series.index
)
portfolio_return_series.name = "Portfolio"
portfolio_return_series.describe()

## 5. Cumulative Portfolio Return

Cumulative return compounds each daily portfolio return through time.

In [ ]:
portfolio_cumulative_return = cumulative_returns(portfolio_return_series)
cumulative_figure = px.line(
    portfolio_cumulative_return,
    title="Cumulative Portfolio Return",
    labels={"index": "Date", "value": "Cumulative Return"},
)
cumulative_figure.update_yaxes(tickformat=".0%")
cumulative_figure

## 6. Rolling Volatility

A 63-trading-day window approximates one quarter. Values are annualized using 252 periods per year.

In [ ]:
portfolio_rolling_volatility = rolling_volatility(
    portfolio_return_series,
    window=63,
)
volatility_figure = px.line(
    portfolio_rolling_volatility,
    title="63-Day Annualized Rolling Volatility",
    labels={"index": "Date", "value": "Annualized Volatility"},
)
volatility_figure.update_yaxes(tickformat=".0%")
volatility_figure

## 7. Drawdown

Drawdown measures the portfolio's percentage decline from its running wealth peak.

In [ ]:
portfolio_drawdown = drawdown_series(portfolio_return_series)
drawdown_figure = px.area(
    portfolio_drawdown,
    title="Portfolio Drawdown",
    labels={"index": "Date", "value": "Drawdown"},
)
drawdown_figure.update_yaxes(tickformat=".0%")
drawdown_figure

## 8. Rolling Beta

Rolling beta estimates the portfolio's changing sensitivity to `SPY` over the same 63-day horizon.

In [ ]:
portfolio_rolling_beta = rolling_beta(
    portfolio_return_series,
    benchmark_return_series,
    window=63,
)
beta_figure = px.line(
    portfolio_rolling_beta,
    title="63-Day Rolling Beta vs SPY",
    labels={"index": "Date", "value": "Beta"},
)
beta_figure.add_hline(y=1.0, line_dash="dash", line_color="gray")
beta_figure

## 9. Return Distribution

The histogram provides a direct view of the portfolio's observed daily-return distribution and tail behavior.

In [ ]:
distribution_figure = px.histogram(
    portfolio_return_series,
    nbins=60,
    title="Portfolio Daily Return Distribution",
    labels={"value": "Daily Return"},
)
distribution_figure.update_xaxes(tickformat=".1%")
distribution_figure

## Interpretation Notes

- Metrics are historical estimates and depend on the selected sample and ETF proxies.
- The portfolio is illustrative, not an investment recommendation.
- Free adjusted-close data can contain missing or revised observations.
- VaR and CVaR describe observed return tails; they do not bound future losses.